# LeetCode 3279 — Maximum Total Area Occupied by Pistons bằng Python

- Link: https://leetcode.com/problems/maximum-total-area-occupied-by-pistons/
- Độ khó: **Hard** · **Premium**
- Chủ đề: **Mô phỏng theo sự kiện**: tổng diện tích là hàm **tuyến tính từng khúc** theo thời gian, độ dốc đổi `±2` mỗi khi một piston chạm đầu; chu kỳ `2·height` ⇒ quét các thời điểm sự kiện trong `[0, 2h]`

> **Ghi chú Premium:** bài Premium nên LeetCode không trả đề qua API công khai. Đề, ví dụ, ràng buộc và chữ ký `maxArea(self, height, positions, directions)` lấy theo trang doocs (https://leetcode.doocs.org/en/lc/3279/) — vui lòng **đối chiếu lại** nếu có tài khoản Premium.

Skeleton (theo doocs):

```python
class Solution:
    def maxArea(self, height: int, positions: List[int], directions: str) -> int:
```

## 1. Mục tiêu học

1. Mỗi piston dao động trong `[0, h]` với vận tốc `±1`, đổi chiều ở hai đầu ⇒ vị trí tuần hoàn chu kỳ `2h`.
2. Tổng diện tích `A(t) = Σ p_i(t)` có độ dốc `= (#đang đi lên) − (#đang đi xuống)`; độ dốc chỉ đổi tại các **thời điểm chạm đầu** — mỗi lần chạm đổi độ dốc đúng `±2`.
3. Mỗi piston chạm đầu đúng **2 lần** trong một chu kỳ ⇒ `2n` sự kiện; sắp xếp rồi quét, cập nhật `A` trên từng đoạn tuyến tính — cực đại của hàm tuyến tính từng khúc đạt tại một điểm đầu mút đoạn.
4. Cách thay thế: mảng hiệu độ dài `2h + 1` (không cần sắp xếp) khi `h` vừa phải.
5. Kiểm chứng bằng mô phỏng từng giây.

## 2. Đề bài & ví dụ

Có các piston với độ cao tối đa `height`. `positions[i]` là vị trí hiện tại của piston `i` (bằng diện tích phía dưới nó), `directions[i] ∈ {'U', 'D'}` là chiều chuyển động. Mỗi giây, mỗi piston di chuyển 1 đơn vị theo chiều hiện tại; khi chạm một đầu (`0` hoặc `height`) thì đổi chiều. Trả về **tổng diện tích lớn nhất** có thể dưới tất cả các piston (tại một thời điểm nào đó, kể cả thời điểm hiện tại).

**Ví dụ 1:** `height = 5`, `positions = [2,5]`, `directions = "UD"` → `7`.
**Ví dụ 2:** `height = 6`, `positions = [0,0,6,3]`, `directions = "UUDU"` → `15` (sau 3 giây: `[3,3,3,6]`).

**Ràng buộc (theo doocs):** `1 ≤ height ≤ 10^6`; `1 ≤ n ≤ 10^5`; `0 ≤ positions[i] ≤ height`.

In [1]:
from typing import List
from collections import defaultdict

examples = [(5, [2, 5], "UD", 7), (6, [0, 0, 6, 3], "UUDU", 15)]
for h, pos, dirs, exp in examples:
    print('height =', h, pos, repr(dirs), '-> kỳ vọng', exp)

height = 5 [2, 5] 'UD' -> kỳ vọng 7
height = 6 [0, 0, 6, 3] 'UUDU' -> kỳ vọng 15


## 3. Kiến thức nền

### 3.1. Mô phỏng một piston

Quy ước: trước khi đi một bước, nếu piston đang ở đầu và chiều hướng ra ngoài thì đổi chiều (ví dụ `p = 0`, `'D'` ⇒ thành `'U'`). Vị trí sau `t` giây có chu kỳ `2h`.

In [2]:
def step_all(h, p, d):
    for i in range(len(p)):
        if p[i] == 0 and d[i] == 'D':
            d[i] = 'U'
        if p[i] == h and d[i] == 'U':
            d[i] = 'D'
        p[i] += 1 if d[i] == 'U' else -1

p, d = [2], ['U']
trace_pos = [p[0]]
for _ in range(12):
    step_all(5, p, d); trace_pos.append(p[0])
print('h = 5, bắt đầu 2 đi lên:', trace_pos, '-> chu kỳ 10 = 2h')

h = 5, bắt đầu 2 đi lên: [2, 3, 4, 5, 4, 3, 2, 1, 0, 1, 2, 3, 4] -> chu kỳ 10 = 2h


### 3.2. Thời điểm chạm đầu của một piston (trong một chu kỳ `[0, 2h]`)

- Đi lên từ `p`: chạm `h` tại `t = h − p` (độ dốc của nó đổi `+1 → −1`, tức tổng độ dốc **−2**), chạm `0` tại `t = 2h − p` (**+2**).
- Đi xuống từ `p`: chạm `0` tại `t = p` (**+2**), chạm `h` tại `t = h + p` (**−2**).

Sau `2h` mọi piston trở lại trạng thái ban đầu ⇒ chỉ cần xét `t ∈ [0, 2h]`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2 (`h = 6`): `A(0) = 9`, độ dốc ban đầu `= 3 − 1 = 2` (ba piston lên, một xuống). Sự kiện: piston 4 (`3`, lên) chạm đỉnh tại `t = 3` (−2); piston 3 (`6`, xuống) chạm đáy tại `t = 6` (+2); piston 1, 2 (`0`, lên) chạm đỉnh tại `t = 6` (−2 mỗi cái)…

- `t ∈ [0, 3]`: dốc `+2` ⇒ `A(3) = 9 + 6 = 15`.
- `t ∈ [3, 6]`: dốc `0` ⇒ `A` giữ `15`.
- Sau `t = 6`: dốc `0 + 2 − 4 = −2` ⇒ giảm.

Cực đại `15` ✓.

## 5. Cách 1 — Brute Force: mô phỏng từng giây trong một chu kỳ

In [3]:
def brute(h, pos, dirs):
    p, d = list(pos), list(dirs)
    best = sum(p)
    for _ in range(2 * h):
        step_all(h, p, d)
        best = max(best, sum(p))
    return best

for h, pos, dirs, exp in examples:
    print(brute(h, pos, dirs), '| kỳ vọng', exp)

7 | kỳ vọng 7
15 | kỳ vọng 15


**Độ phức tạp:** `O(n · h)` — tới `2·10^{11}` với ràng buộc lớn.

## 6. Ý tưởng thuật toán / bước đột phá

```
A = Σ positions; slope = #U − #D
mỗi piston thêm 2 sự kiện (thời điểm, Δslope) như mục 3.2
sắp sự kiện theo thời điểm; prev = 0
for (t, Δ): A += (t − prev)·slope; prev = t; slope += Δ; ans = max(ans, A)
```

`O(n log n)`. Gộp các sự kiện cùng thời điểm (dùng từ điển) để chỉ cập nhật `A` một lần mỗi thời điểm.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đánh giá `A(t)` tại mọi thời điểm sự kiện bằng công thức vị trí

Vị trí piston tại `t` có công thức đóng (phản xạ trong `[0, h]`). Cực đại của `A` đạt tại một thời điểm sự kiện (hoặc `t = 0`) ⇒ đánh giá `A` tại `O(n)` thời điểm, mỗi lần `O(n)` ⇒ `O(n²)`.

In [4]:
def position_at(h, p, d, t):
    """Vị trí sau t giây của piston (p, d)."""
    u = p if d == 'U' else 2 * h - p           # "tọa độ mở" trên vòng tròn độ dài 2h, đi lên tại u ∈ [0, h]
    u = (u + t) % (2 * h)
    return u if u <= h else 2 * h - u

def event_eval(h, pos, dirs):
    times = {0}
    for p, d in zip(pos, dirs):
        if d == 'U':
            times |= {h - p, 2 * h - p}
        else:
            times |= {p, h + p}
    return max(sum(position_at(h, p, d, t) for p, d in zip(pos, dirs)) for t in times if 0 <= t <= 2 * h)

for h, pos, dirs, exp in examples:
    print(event_eval(h, pos, dirs), '| kỳ vọng', exp)

7 | kỳ vọng 7
15 | kỳ vọng 15


Ý tưởng "tọa độ mở": duỗi đường đi qua lại thành chuyển động tròn trên vòng độ dài `2h` (`u ∈ [0, h]` ứng với pha đi lên, `u ∈ (h, 2h)` ứng với pha đi xuống ở độ cao `2h − u`). Piston ở `p` đi xuống ứng với `u = 2h − p`.

### 7.2. Cách 3 — Quét sự kiện với từ điển hiệu (lời giải chính)

In [5]:
def sweep(h, pos, dirs):
    delta = defaultdict(int)
    A = slope = 0
    for p, d in zip(pos, dirs):
        A += p
        if d == 'U':
            slope += 1
            delta[h - p] -= 2
            delta[2 * h - p] += 2
        else:
            slope -= 1
            delta[p] += 2
            delta[h + p] -= 2
    best, prev = A, 0
    for t in sorted(delta):
        A += (t - prev) * slope
        prev = t
        slope += delta[t]
        best = max(best, A)
    return best

for h, pos, dirs, exp in examples:
    print(sweep(h, pos, dirs), '| kỳ vọng', exp)

7 | kỳ vọng 7
15 | kỳ vọng 15


### 7.3. Cách 4 — Mảng hiệu độ dài `2h + 1` (không sắp xếp)

Khi `h ≤ 10^6`, mảng `2·10^6` phần tử vẫn chấp nhận được; quét `t = 0..2h` tích lũy độ dốc — `O(n + h)`.

In [6]:
def diff_array(h, pos, dirs):
    D = [0] * (2 * h + 1)
    A = slope = 0
    for p, d in zip(pos, dirs):
        A += p
        if d == 'U':
            slope += 1; D[h - p] -= 2; D[2 * h - p] += 2
        else:
            slope -= 1; D[p] += 2; D[h + p] -= 2
    slope += D[0]                               # sự kiện ngay tại t = 0
    best = A
    for t in range(1, 2 * h + 1):
        A += slope
        best = max(best, A)
        slope += D[t]
    return best

for h, pos, dirs, exp in examples:
    print(diff_array(h, pos, dirs), '| kỳ vọng', exp)

7 | kỳ vọng 7
15 | kỳ vọng 15


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 2.

In [7]:
def trace(h, pos, dirs):
    delta = defaultdict(int); A = slope = 0
    for p, d in zip(pos, dirs):
        A += p
        if d == 'U':
            slope += 1; delta[h - p] -= 2; delta[2 * h - p] += 2
        else:
            slope -= 1; delta[p] += 2; delta[h + p] -= 2
    print(f'A(0) = {A}, độ dốc ban đầu = {slope}, sự kiện = {dict(sorted(delta.items()))}')
    best, prev = A, 0
    for t in sorted(delta):
        A += (t - prev) * slope
        print(f'  t = {t:2d}: A = {A:3d} (dốc {slope:+d} trên [{prev}, {t}]) -> dốc mới {slope + delta[t]:+d}')
        prev = t; slope += delta[t]; best = max(best, A)
    print('đáp án:', best, '| mô phỏng:', brute(h, pos, dirs))

trace(6, [0, 0, 6, 3], "UUDU")

A(0) = 9, độ dốc ban đầu = 2, sự kiện = {3: -2, 6: -2, 9: 2, 12: 2}
  t =  3: A =  15 (dốc +2 trên [0, 3]) -> dốc mới +0
  t =  6: A =  15 (dốc +0 trên [3, 6]) -> dốc mới -2
  t =  9: A =   9 (dốc -2 trên [6, 9]) -> dốc mới +0
  t = 12: A =   9 (dốc +0 trên [9, 12]) -> dốc mới +2
đáp án: 15 | mô phỏng: 15


## 9. Code nộp LeetCode

(Chữ ký theo doocs — xem ghi chú Premium.)

In [8]:
class Solution:
    def maxArea(self, height: int, positions: List[int], directions: str) -> int:
        delta = defaultdict(int)
        area = slope = 0
        h = height
        for p, d in zip(positions, directions):
            area += p
            if d == 'U':
                slope += 1
                delta[h - p] -= 2
                delta[2 * h - p] += 2
            else:
                slope -= 1
                delta[p] += 2
                delta[h + p] -= 2
        best = area
        prev = 0
        for t in sorted(delta):
            area += (t - prev) * slope
            prev = t
            slope += delta[t]
            if area > best:
                best = area
        return best

for h, pos, dirs, exp in examples:
    got = Solution().maxArea(h, pos, dirs)
    print(got, 'OK' if got == exp else 'SAI')

7 OK
15 OK


## 10. Vì sao thuật toán đúng

1. **Tuyến tính từng khúc:** giữa hai thời điểm chạm đầu liên tiếp (của bất kỳ piston nào), mọi vận tốc không đổi ⇒ `A(t)` tuyến tính với độ dốc `= #lên − #xuống`.
2. **Đổi độ dốc:** một piston đổi `+1 → −1` (chạm đỉnh, Δ = −2) hoặc `−1 → +1` (chạm đáy, Δ = +2). Các thời điểm như mục 3.2 (piston ở đầu nhưng hướng ra ngoài được xem như chạm ngay tại `t = 0`, Δ đặt ở `t = 0`).
3. **Chu kỳ:** sau `2h` giây mọi piston về trạng thái ban đầu ⇒ `max_{t ≥ 0} A(t) = max_{t ∈ [0, 2h]} A(t)`.
4. **Cực đại tại đầu mút:** hàm tuyến tính trên một đoạn đạt max tại một đầu mút ⇒ chỉ cần xét `t = 0` và các thời điểm sự kiện. Thời điểm sự kiện là số nguyên nên trùng với thời điểm "mỗi giây" của đề.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng | `O(n h)` | `O(n)` |
| 2. Đánh giá tại sự kiện | `O(n²)` | `O(n)` |
| 3. Quét sự kiện | `O(n log n)` | `O(n)` |
| 4. Mảng hiệu | `O(n + h)` | `O(h)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'event_eval': event_eval, 'sweep': sweep, 'diff_array': diff_array,
         'submit': lambda h, p, d: Solution().maxArea(h, p, d)}
edge = [(1, [0], "U", 1), (1, [0], "D", 1), (1, [1], "U", 1), (3, [3, 0], "UD", 3), (4, [2, 2], "UD", 4),
        (4, [4, 1, 2, 1], "UDDU", None)]
for h, pos, dirs, exp in examples + edge:
    e = brute(h, pos, dirs)
    if exp is not None:
        assert e == exp, (h, pos, dirs, e)
    for name, f in impls.items():
        assert f(h, pos, dirs) == e, (name, h, pos, dirs)
print('Ví dụ đề và ca biên: tất cả đúng. Ca (4, [4,1,2,1], "UDDU") ->', brute(4, [4, 1, 2, 1], "UDDU"))

Ví dụ đề và ca biên: tất cả đúng. Ca (4, [4,1,2,1], "UDDU") -> 10


### 12.2. Đối chiếu ngẫu nhiên với mô phỏng

In [10]:
import random
rng = random.Random(3279)
cnt = 0
for _ in range(1500):
    h = rng.randint(1, 8); n = rng.randint(1, 6)
    pos = [rng.randint(0, h) for _ in range(n)]
    dirs = ''.join(rng.choice('UD') for _ in range(n))
    exp = brute(h, pos, dirs)
    for name, f in impls.items():
        assert f(h, pos, dirs) == exp, (name, h, pos, dirs)
    cnt += 1
print('Khớp mô phỏng trên', cnt, 'bộ test (kể cả piston ở đầu hướng ra ngoài).')

Khớp mô phỏng trên 1500 bộ test (kể cả piston ở đầu hướng ra ngoài).


### 12.3. Ca lớn `n = 10^5`, `height = 10^6`

In [11]:
import time
H, N = 10 ** 6, 10 ** 5
big = {
    'ngẫu nhiên': (H, [rng.randint(0, H) for _ in range(N)], ''.join(rng.choice('UD') for _ in range(N))),
    'tất cả lên từ 0': (H, [0] * N, 'U' * N),
    'nửa ở đỉnh đi xuống': (H, [H] * (N // 2) + [0] * (N // 2), 'D' * (N // 2) + 'U' * (N // 2)),
}
for name, (h, pos, dirs) in big.items():
    t = time.time(); a = Solution().maxArea(h, pos, dirs); t1 = time.time() - t
    t = time.time(); b = diff_array(h, pos, dirs); t2 = time.time() - t
    assert a == b
    print(f"{name:20s}: {a:14d} | nộp {t1:.2f}s | mảng hiệu {t2:.2f}s")
assert Solution().maxArea(H, [0] * N, 'U' * N) == H * N
print('tất cả lên từ 0: max = H·N tại t = H — đúng.')

ngẫu nhiên          :    50151021966 | nộp 0.13s | mảng hiệu 0.18s
tất cả lên từ 0     :   100000000000 | nộp 0.02s | mảng hiệu 0.17s


nửa ở đỉnh đi xuống :    50000000000 | nộp 0.02s | mảng hiệu 0.15s
tất cả lên từ 0: max = H·N tại t = H — đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ xét `t ∈ [0, h]` (nửa chu kỳ)

In [12]:
def wrong_half_period(h, pos, dirs):
    p, d = list(pos), list(dirs); best = sum(p)
    for _ in range(h):                                         # SAI: chu kỳ là 2h
        step_all(h, p, d); best = max(best, sum(p))
    return best

bad = (4, [4, 1, 2, 1], "UDDU")
print('sai :', wrong_half_period(*bad), '| đúng:', Solution().maxArea(*bad))

sai : 8 | đúng: 10


**Vì sao sai:** các piston lệch pha nhau; trạng thái tốt nhất có thể xuất hiện ở nửa sau chu kỳ.

### Lỗi 2 — Kẹp ở biên (không đổi chiều)

In [13]:
def wrong_clamp(h, pos, dirs):
    p = list(pos); best = sum(p)
    for _ in range(2 * h):
        p = [min(h, x + 1) if d == 'U' else max(0, x - 1) for x, d in zip(p, dirs)]   # SAI
        best = max(best, sum(p))
    return best

print('sai :', wrong_clamp(1, [0], "D"), '| đúng:', Solution().maxArea(1, [0], "D"))
print('sai :', wrong_clamp(4, [1, 3], "DD"), '| đúng:', Solution().maxArea(4, [1, 3], "DD"))

sai : 0 | đúng: 1
sai : 4 | đúng: 6


**Vì sao sai:** piston chạm đầu phải **đổi chiều**; kẹp lại thì piston đi xuống nằm mãi ở đáy (`h = 1`, `[0]`, `'D'`: đúng ra nó đi lên tới 1). Với nhiều piston cùng đi xuống, bản sai không bao giờ thấy chúng quay lên.

### Lỗi 3 — Mỗi lần chạm đầu chỉ đổi độ dốc `±1`

In [14]:
def wrong_delta_one(h, pos, dirs):
    delta = defaultdict(int); A = slope = 0
    for p, d in zip(pos, dirs):
        A += p
        if d == 'U':
            slope += 1; delta[h - p] -= 1; delta[2 * h - p] += 1        # SAI: phải là ±2
        else:
            slope -= 1; delta[p] += 1; delta[h + p] -= 1
    best, prev = A, 0
    for t in sorted(delta):
        A += (t - prev) * slope; prev = t; slope += delta[t]; best = max(best, A)
    return best

print('sai :', wrong_delta_one(*examples[1][:3]), '| đúng:', Solution().maxArea(*examples[1][:3]))

sai : 21 | đúng: 15


**Vì sao sai:** vận tốc đổi từ `+1` sang `−1` — chênh lệch **2**.

### Lỗi 4 — Cập nhật độ dốc **trước** khi cộng đoạn

In [15]:
def wrong_order(h, pos, dirs):
    delta = defaultdict(int); A = slope = 0
    for p, d in zip(pos, dirs):
        A += p
        if d == 'U':
            slope += 1; delta[h - p] -= 2; delta[2 * h - p] += 2
        else:
            slope -= 1; delta[p] += 2; delta[h + p] -= 2
    best, prev = A, 0
    for t in sorted(delta):
        slope += delta[t]                                            # SAI: đổi dốc quá sớm
        A += (t - prev) * slope; prev = t; best = max(best, A)
    return best

print('sai :', wrong_order(*examples[1][:3]), '| đúng:', Solution().maxArea(*examples[1][:3]))

sai : 9 | đúng: 15


**Vì sao sai:** trên đoạn `[prev, t]` vẫn là độ dốc **cũ**; sự kiện tại `t` chỉ ảnh hưởng từ `t` trở đi.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Nhiều vật chuyển động đều, đổi vận tốc ở các thời điểm rời rạc ⇒ **hàm tuyến tính từng khúc** + quét sự kiện (sweep line theo thời gian).
- Chuyển động qua lại trong đoạn ⇒ "duỗi" thành chuyển động tròn chu kỳ `2h` (tọa độ mở).
- Cực đại/cực tiểu hàm tuyến tính từng khúc nằm ở điểm gãy.

## 15. Biến thể của bài

1. Diện tích **nhỏ nhất** ⇒ `min` thay `max`.
2. Thời điểm **sớm nhất** đạt cực đại.
3. Piston có vận tốc khác nhau `v_i` ⇒ chu kỳ `lcm`, sự kiện nhiều hơn; vẫn tuyến tính từng khúc.
4. Chỉ xét trong `T` giây đầu ⇒ cắt sự kiện tại `T` (có thể `T < 2h`).

In [16]:
# Biến thể 1, 2, 4
def events(h, pos, dirs):
    delta = defaultdict(int); A = slope = 0
    for p, d in zip(pos, dirs):
        A += p
        if d == 'U':
            slope += 1; delta[h - p] -= 2; delta[2 * h - p] += 2
        else:
            slope -= 1; delta[p] += 2; delta[h + p] -= 2
    return A, slope, delta

def min_area(h, pos, dirs):
    A, slope, delta = events(h, pos, dirs)
    best, prev = A, 0
    for t in sorted(delta):
        A += (t - prev) * slope; prev = t; slope += delta[t]; best = min(best, A)
    return best

def earliest_max(h, pos, dirs):
    target = Solution().maxArea(h, pos, dirs)
    A, slope, delta = events(h, pos, dirs)
    if A == target:
        return 0
    prev = 0
    for t in sorted(delta):
        A += (t - prev) * slope; prev = t; slope += delta[t]
        if A == target:
            return t

def within_T(h, pos, dirs, T):
    A, slope, delta = events(h, pos, dirs)
    best, prev = A, 0
    for t in sorted(set(delta) | {T}):
        if t > T:
            break
        A += (t - prev) * slope; prev = t
        slope += delta.get(t, 0)
        best = max(best, A)
    return best

rng = random.Random(15)
for _ in range(400):
    h = rng.randint(1, 7); n = rng.randint(1, 5)
    pos = [rng.randint(0, h) for _ in range(n)]; dirs = ''.join(rng.choice('UD') for _ in range(n))
    p, d = list(pos), list(dirs); seq = [sum(p)]
    for _ in range(2 * h):
        step_all(h, p, d); seq.append(sum(p))
    assert min_area(h, pos, dirs) == min(seq)
    assert earliest_max(h, pos, dirs) == seq.index(max(seq))
    T = rng.randint(0, 2 * h)
    assert within_T(h, pos, dirs, T) == max(seq[:T + 1])
print('Biến thể min, thời điểm sớm nhất, giới hạn T: khớp mô phỏng.')

Biến thể min, thời điểm sớm nhất, giới hạn T: khớp mô phỏng.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mô phỏng từng giây | `O(n h)` | Oracle |
| 2 | Đánh giá tại sự kiện | `O(n²)` | Công thức tọa độ mở |
| 3 | Quét sự kiện | `O(n log n)` | **Nộp bài** |
| 4 | Mảng hiệu | `O(n + h)` | Khi `h` vừa phải |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `height = 3`, `positions = [1]`, `directions = "D"`.

**Bài 2 (tính tay):** `height = 2`, `positions = [0, 2]`, `directions = "UU"`.

**Bài 3 (điền TODO):** trả về **vị trí các piston** tại thời điểm sớm nhất đạt cực đại.

**Bài 4 (tự viết lại):** biến thể 3 — piston `i` có vận tốc `v_i ∈ {1, 2}`.

**Bài 5 (mini challenge):** số **thời điểm nguyên** trong `[0, 2h)` đạt cực đại.

In [17]:
def positions_at_max_todo(h, pos, dirs):
    # TODO
    raise NotImplementedError

def two_speeds_todo(h, pos, dirs, v):
    # TODO
    raise NotImplementedError

def count_max_times_todo(h, pos, dirs):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Một piston, sẽ lên tới đỉnh. (2) `max = h`. (3) `3`.

**Bài 2.** (1) Piston 2 ở đỉnh hướng lên ⇒ đi xuống ngay. (2) Hai piston ngược pha: tổng luôn `2`. (3) `2`.

**Bài 3.** (1) `earliest_max`. (2) `position_at` cho từng piston. (3) Tổng phải bằng cực đại.

**Bài 4.** (1) Chu kỳ chung `2h` vẫn đúng? Với `v = 2`, chu kỳ riêng `h` — chung là `2h`. (2) Mô phỏng hoặc sự kiện với Δ dốc `±2v`. (3) Thời điểm chạm có thể không nguyên với `h` lẻ — cẩn thận (dùng thời gian nhân đôi).

**Bài 5.** (1) Trên mỗi đoạn dốc `0` đạt max, mọi điểm nguyên đều đạt. (2) Cộng độ dài các đoạn đó + các điểm cô lập. (3) Mô phỏng đối chiếu.

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2, 3
assert Solution().maxArea(3, [1], "D") == 3
assert Solution().maxArea(2, [0, 2], "UU") == 2 == brute(2, [0, 2], "UU")

def positions_at_max(h, pos, dirs):
    t = earliest_max(h, pos, dirs)
    return t, [position_at(h, p, d, t) for p, d in zip(pos, dirs)]

t_, ps = positions_at_max(6, [0, 0, 6, 3], "UUDU")
assert sum(ps) == 15
print('Bài 1: 3 | Bài 2: 2 | Bài 3: t =', t_, 'vị trí', ps)

Bài 1: 3 | Bài 2: 2 | Bài 3: t = 3 vị trí [3, 3, 3, 6]


In [19]:
# Bài 4, 5
def two_speeds(h, pos, dirs, v):
    p, d = list(pos), list(dirs); best = sum(p)
    for _ in range(2 * h):
        for i in range(len(p)):
            for _ in range(v[i]):
                if p[i] == 0 and d[i] == 'D':
                    d[i] = 'U'
                if p[i] == h and d[i] == 'U':
                    d[i] = 'D'
                p[i] += 1 if d[i] == 'U' else -1
        best = max(best, sum(p))
    return best

def count_max_times(h, pos, dirs):
    target = Solution().maxArea(h, pos, dirs)
    return sum(1 for t in range(2 * h) if sum(position_at(h, p, d, t) for p, d in zip(pos, dirs)) == target)

rng = random.Random(4)
for _ in range(200):
    h = rng.randint(1, 6); n = rng.randint(1, 4)
    pos = [rng.randint(0, h) for _ in range(n)]; dirs = ''.join(rng.choice('UD') for _ in range(n))
    p, d = list(pos), list(dirs); seq = [sum(p)]
    for _ in range(2 * h - 1):
        step_all(h, p, d); seq.append(sum(p))
    assert count_max_times(h, pos, dirs) == seq.count(max(seq))
    assert two_speeds(h, pos, dirs, [1] * n) == brute(h, pos, dirs)
print('Bài 4 (v = 1 trùng bài gốc), Bài 5 khớp mô phỏng. Ví dụ two_speeds:', two_speeds(5, [2, 5], "UD", [2, 1]))

Bài 4 (v = 1 trùng bài gốc), Bài 5 khớp mô phỏng. Ví dụ two_speeds: 8


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
A = Σ pos; slope = #U − #D
U tại p: delta[h−p] −= 2; delta[2h−p] += 2
D tại p: delta[p]   += 2; delta[h+p]  −= 2
for t sorted: A += (t − prev)·slope; prev = t; slope += delta[t]; ans = max(ans, A)
```

**Checklist:**

- [ ] Xét đủ một chu kỳ `2h`?
- [ ] Chạm đầu ⇒ đổi chiều (không kẹp)?
- [ ] Δ độ dốc `±2`?
- [ ] Cộng đoạn với độ dốc **cũ** rồi mới cập nhật?
- [ ] Piston ở đầu hướng ra ngoài: sự kiện tại `t = 0`?